# Study report: zephyr-versions

The study measures the three latencies (IRQ entry, ISR to thread, thread
to thread) and their growth with the load on each Zephyr version (each
tree). Each figure gives one colour to each tree, so the versions are
easy to compare. Now the study has one tree (ws-up, v4.5.0-rc3-40): the
figures show the baseline for the next versions.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters: the instance folder. Set LATBENCH_INSTANCE (study.py analysis notebooks does), or change the default.
import os
import subprocess
import sys
from pathlib import Path

INSTANCE = Path(os.environ.get(
    "LATBENCH_INSTANCE",
    "/wrk/z/lab-kernel-latency/results/kernel-latency/st-zephyr-versions@nucleo_g474re"))
# LATBENCH_REPO: set by "study.py analysis notebooks"; else the repo of the current folder.
REPO = Path(os.environ.get("LATBENCH_REPO") or subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Trees

The Zephyr version of each tree, as the images print it (LAT META
`zephyr`).

In [ ]:
EXPERIMENTS = ["irq-baseline-1m", "wake-anchors-1m", "thread-baseline-1m", "thread-switch", "wake-systick-prio-1m",
               "wake-timeout-backends-1m", "wake-threads-1m", "wake-timers-1m", "wake-clock-drift",
               "wake-clock-drift-layout"]
shown = [e for e in EXPERIMENTS if nbfig.built(inst, e)]
pd.concat([inst.meta(e)[["tree", "zephyr"]].assign(experiment=e) for e in shown]).drop_duplicates() \
    .groupby(["tree", "zephyr"]).experiment.apply(", ".join).to_frame()

## M1: IRQ entry (irq-baseline-1m)

The edge to the first instruction of the ISR, for each interrupt path,
with an idle CPU: `stock` (the ISR table), `wrapped` (`_isr_wrapper`),
`direct` (ISR_DIRECT) and `zli` (zero latency interrupt).

In [ ]:
df = inst.rows("irq-baseline-1m")
nbfig.dots(df, "IRQ entry per interrupt path").show()
nbfig.pivot(df, ["p50", "p99", "max"])

## M2: ISR to thread (wake-anchors-1m)

The ISR gives a semaphore and the woken thread runs. The anchors:
`default` (time slicing on), `noslice` (TIMESLICING=n), `lean` (`common/conf/lean.conf`: no time
slicing, no stack guard, -O2); `legacy` is the PendSV switch, `switch`
is USE_SWITCH.

In [ ]:
df = inst.rows("wake-anchors-1m")
nbfig.dots(df, "ISR to thread per anchor", stats=("p50", "p99", "p99.99", "max")).show()
nbfig.ccdf(nbfig.hists_of(inst, "wake-anchors-1m"), "wake-anchors-1m: CCDF").show()

In [ ]:
nbfig.pivot(df, "p50")

## M3: thread to thread (thread-baseline-1m, thread-switch)

The wake call in the giver thread to the first instruction of the taker,
for each kernel primitive. `-nopreempt`: the taker has a lower priority
than the giver, so it runs when the giver blocks again.

In [ ]:
df = inst.rows("thread-baseline-1m")
nbfig.dots(df, "Thread to thread per primitive").show()
nbfig.pivot(df, "p50")

In [ ]:
df = inst.rows("thread-switch")
nbfig.dots(df, "Thread to thread: legacy switch against USE_SWITCH").show()
nbfig.pivot(df, "p50")

## Growth with the load

### Kernel timers (wake-timers-1m)

The p50, p99 and max over the number of periodic load timers, with time
slicing (`default`) and without (`noslice`). The points `t256far` (256
timers far in the future) are in the table only.

In [ ]:
df = inst.rows("wake-timers-1m")
df["series"] = df["tree"] + " " + df["axis.anchor"]
near = df[~df.point.str.startswith("t256far")]
nbfig.sweep(near, "value.timers", "ISR to thread over the load timers", group="series").show()
nbfig.pivot(df, "p50")

### Load threads (wake-threads-1m)

The p50 over the number of load threads of each kind: `blocked` (wait
on a semaphore of its own), `sleeper` (sleeps for a random time, in a
loop), `busy` (a busy loop at the lowest application priority).

In [ ]:
df = inst.rows("wake-threads-1m")
base = df[df["axis.kind"] == "none"]
df = pd.concat([df[df["axis.kind"] != "none"]] + [base.assign(**{"axis.kind": k}) for k in ("blocked", "sleeper", "busy")])
df["series"] = df["tree"] + " " + df["axis.kind"]
nbfig.sweep(df, "value.threads", "ISR to thread over the load threads", stats=("p50", "p99"), group="series").show()

### Timeout queue backends and SysTick priority

`wake-timeout-backends-1m`: the backends of the timeout queue with the
same load. `wake-systick-prio-1m`: SysTick at the priority of the device
IRQ (`same`) or one level lower (`lower`), with 64 and 256 load timers.

In [ ]:
df = inst.rows("wake-timeout-backends-1m")
nbfig.dots(df, "Timeout queue backends").show()
nbfig.pivot(df, "p50")

In [ ]:
df = inst.rows("wake-systick-prio-1m")
nbfig.dots(df, "SysTick priority", stats=("p50", "p99", "p99.99", "max")).show()
nbfig.pivot(df, "max")

## Clock drift (wake-clock-drift, wake-clock-drift-layout)

The kernel cycle counter against the stimulus timer (the same clock):
`drift/wake` is LAT META `clock_drift_cyc` / samples, `ppm` is
`clock_drift_cyc` / `clock_elapsed_cyc`.

In [ ]:
import plotly.graph_objects as go

drift = inst.drift([e for e in ("wake-clock-drift", "wake-clock-drift-layout") if nbfig.built(inst, e)]).rename(
    columns={"drift_per_sample": "drift/wake", "drift_ppm": "ppm"})
fig = go.Figure()
for k, t in enumerate(dict.fromkeys(drift.tree)):
    sub = drift[drift.tree == t]
    fig.add_trace(go.Bar(x=sub.experiment + " " + sub.point, y=sub["drift/wake"], name=t, marker_color=nbfig.color(k),
                         hovertemplate="%{x}: %{y:.3f} cycles per wake<extra>" + t + "</extra>"))
nbfig.style(fig, "Cycles lost per wake", "point", "clock_drift_cyc / samples").show()
drift[["experiment", "tree", "point", "samples", "clock_drift_cyc", "drift/wake", "ppm"]]